In [3]:
from FLAlgorithms.trainmodel.generator import Generator
import torch
from torch import nn, optim
import torch.nn.functional as F
from thop import profile


In [46]:
from FLAlgorithms.trainmodel.models import Net

generator_model = Generator('flags-regression', problem_type='regression', n_class=30, embedding=1)
cnn_model = Net('flags-regression', model='cnn', problem_type='regression', output_dimension=30)
lstm_model = Net('flags-regression', model='lstm', problem_type='regression', output_dimension=30)

# Initialize optimizer
optimizer = optim.SGD(generator_model.parameters(), lr=0.001, momentum=0.9)

    
torch.save(generator_model.state_dict(), 'generator_model.pt')


Dataset flags-regression
Build layer 64 X 256
Build last layer 256 X 32
Creating model for flags-regression
Network configs: [6, 16, 'F']
Creating model for flags-regression
Network configs: ['LSTM-480', 'F']


In [24]:
generator_model

Generator(
  (crossentropy_loss): NLLLoss()
  (l1_loss): L1Loss()
  (diversity_loss): DiversityLoss(
    (cosine): CosineSimilarity()
  )
  (dist_loss): MSELoss()
  (embedding_layer): Embedding(30, 32)
  (regression_input_embedding): Linear(in_features=30, out_features=32, bias=True)
  (fc_layers): ModuleList(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (representation_layer): Linear(in_features=256, out_features=32, bias=True)
)

In [40]:
input = torch.randn(32, 30)
macs, params = profile(generator_model, inputs=(input, ))
print(f'Generator: MACs={macs}, Params={params}')

[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm1d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
Generator: MACs=1699840.0, Params=26368.0


In [41]:
input = torch.randn(32, 1, 60 ,6)
macs, params = profile(cnn_model, inputs=(input, ))
print(f'CNN: MACs={macs}, Params={params}')

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
CNN: MACs=6551040.0, Params=17366.0


In [42]:
input = torch.randn(32, 60 ,6)
macs, params = profile(lstm_model, inputs=(input, ))
print(f'LSTM: MACs={macs}, Params={params}')

[INFO] Register count_lstm() for <class 'torch.nn.modules.rnn.LSTM'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
LSTM: MACs=3613716480.0, Params=953342.0
